# 01 · Build the modeling table

**Output:** `data/processed/jobs_model.parquet` — one row per Tennessee job posting with
- the salary target (`log_pay`, `annual_mid`) and the disclosure treatment (`has_salary`),
- the survival target (`duration_days`, `event`; open postings are right-censored at `LAST_CHECKED`),
- employer / location / time / title features.

Also writes `salary_tn.parquet` (the ~375k salary rows for TN jobs) so the 2.5 GB salary file is only scanned once.
Set `SAMPLE_FRAC` (env var or edit below) to e.g. `0.05` for a quick test; sampled outputs get a `_sampleNN` suffix.

In [ ]:
import sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from src import features as F

cfg = F.get_config()
SAMPLE_FRAC = cfg['SAMPLE_FRAC']      # set < 1.0 for a quick trial run (e.g. 0.05)
FORCE_REBUILD = cfg['FORCE_REBUILD']  # True = ignore cached outputs and recompute
SUFFIX = F.sample_suffix(cfg)
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 180)
print('root:', ROOT, '| SAMPLE_FRAC:', SAMPLE_FRAC, '| FORCE_REBUILD:', FORCE_REBUILD)

## 1. Load job records

In [ ]:
t0 = time.time()
jobs_path = F.find_raw_file(F.JOBS_FILE, cfg)
jobs = F.load_jobs(jobs_path, sample_frac=SAMPLE_FRAC)
print(f'{len(jobs):,} job rows from {jobs_path.name} in {time.time()-t0:.0f}s')

filters = [('loaded', len(jobs))]
jobs = jobs[jobs['STATE'] == 'TN']
filters.append(('STATE == TN', len(jobs)))
jobs = jobs[jobs['CREATED'].notna() & jobs['LAST_CHECKED'].notna()]
filters.append(('has CREATED & LAST_CHECKED', len(jobs)))
assert jobs['JOB_HASH'].is_unique
jobs.head(3)

## 2. Extract the salary rows for these jobs (cached)
Streams the salary CSV in 2M-row chunks and keeps only matching `JOB_HASH`es. Takes ~4–8 min the first time (longer from `.gz`).

In [ ]:
sal_path = cfg['DATA_PROC'] / f'salary_tn{SUFFIX}.parquet'
if sal_path.exists() and not FORCE_REBUILD:
    sal = pd.read_parquet(sal_path)
    print(f'loaded cached {sal_path.name}: {len(sal):,} rows')
else:
    t0 = time.time()
    sal = F.extract_salary_for(F.find_raw_file(F.SALARY_FILE, cfg), jobs['JOB_HASH'])
    sal.to_parquet(sal_path, index=False)
    print(f'saved {sal_path.name} in {time.time()-t0:.0f}s')
sal = sal[sal['JOB_HASH'].isin(set(jobs['JOB_HASH']))]
print(f'share of jobs with a salary: {len(sal)/len(jobs):.1%}')
sal.head(3)

## 3. Build features and targets

In [ ]:
t0 = time.time()
df = F.build_model_table(jobs, sal)
df = df[df['duration_days'] > 0]
filters.append(('duration > 0', len(df)))
df = F.finalize_types(df)
print(f'built {df.shape} in {time.time()-t0:.0f}s')
pd.DataFrame(filters, columns=['step', 'rows'])

## 4. Sanity report

In [ ]:
report = {
    'jobs': len(df),
    'has_salary share': df['has_salary'].mean(),
    'pay in 10k-500k (of salaried)': df.loc[df['has_salary'] == 1, 'pay_in_range'].mean(),
    'training rows for salary model (log_pay notna)': int(df['log_pay'].notna().sum()),
    'censored share (still open)': 1 - df['event'].mean(),
    'median duration, salaried (days)': df.loc[df['has_salary'] == 1, 'duration_days'].median(),
    'median duration, no salary (days)': df.loc[df['has_salary'] == 0, 'duration_days'].median(),
    'repost share': df['is_repost'].mean(),
    'unique titles': df['title_clean'].nunique(),
    'unique companies': df['COMPANY_ID'].nunique(),
    'unique ATS domains': df['ats_domain'].nunique(),
    'CREATED range': f"{df['CREATED'].min():%Y-%m-%d} .. {df['CREATED'].max():%Y-%m-%d}",
}
pd.Series(report, name='value').to_frame()

In [ ]:
print(df['pay_frequency'].value_counts(), '\n')
print(df['created_quarter'].value_counts().sort_index(), '\n')
print(df['ats_domain'].value_counts().head(15))

In [ ]:
# Disclosure rate by top title flags / quarter (first look at confounding for Phase 5)
flag_cols = [c for c in df.columns if c.startswith('title_') and df[c].dtype == 'int8' and c != 'title_level']
rows = [(c, int(df[c].sum()), df.loc[df[c] == 1, 'has_salary'].mean()) for c in flag_cols]
display(pd.DataFrame(rows, columns=['flag', 'n', 'salary disclosed share']).sort_values('n', ascending=False))
df.groupby('created_quarter', observed=True)['has_salary'].mean().rename('disclosure rate').to_frame().T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
df.loc[df['pay_in_range'], 'annual_mid'].plot.hist(bins=80, ax=axes[0])
axes[0].set(title='Annual pay (USD, 10k–500k)', xlabel='USD')
np.log10(df['duration_days']).plot.hist(bins=80, ax=axes[1])
axes[1].set(title='Posting duration', xlabel='log10(days)')
for flag, g in df.groupby('has_salary'):
    s = np.sort(g['duration_days'].clip(upper=365).to_numpy())
    axes[2].plot(s, np.arange(1, len(s) + 1) / len(s), label=f'has_salary={flag}')
axes[2].set(title='Duration ECDF (naive, ignores censoring)', xlabel='days (clipped at 365)')
axes[2].legend()
plt.tight_layout(); fig.savefig(cfg['FIG_DIR'] / f'01_overview{SUFFIX}.png', dpi=150); plt.show()

## 5. Save

In [ ]:
out_path = cfg['DATA_PROC'] / f'jobs_model{SUFFIX}.parquet'
df.to_parquet(out_path, index=False)
t0 = time.time()
check = pd.read_parquet(out_path)
print(f'saved {out_path.name}: {check.shape}, {out_path.stat().st_size/1e6:.0f} MB, reload {time.time()-t0:.1f}s')
check.dtypes.value_counts()